# Chapitre 12 · Survivre à Colab (solutions des exercices)

Ce notebook contient **uniquement les réponses aux quatre exercices** du notebook du
chapitre. Le code de la leçon, lui, vit dans le notebook du chapitre et dans le livre.

Si tu n'as pas encore vraiment essayé les exercices, referme ceci : le pacte
« IA débranchée » vaut aussi pour les corrigés.

La première cellule reprend le minimum de la leçon (corpus, modèle, `construire`,
`entrainer`, et la course de référence de 60 pas) pour que chaque validation
s'exécute de façon autonome.

In [ ]:
# Mise en place (reprise de la leçon) : le minimum pour que les validations tournent.
import os
import json
import time
import random
import shutil

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

# Un extrait de fables, répété pour donner de la matière à apprendre.
# (Le vrai notebook du chapitre 10 embarque les 30 fables complètes.)
fable = """\
LA CIGALE ET LA FOURMI
La cigale, ayant chante
Tout l'ete,
Se trouva fort depourvue
Quand la bise fut venue :
Pas un seul petit morceau
De mouche ou de vermisseau.
Elle alla crier famine
Chez la fourmi, sa voisine,
La priant de lui preter
Quelque grain pour subsister
Jusqu'a la saison nouvelle.
"""
corpus = fable * 40

chars = sorted(set(corpus))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
data = torch.tensor([stoi[c] for c in corpus])

block_size = 32
d_model, n_heads, n_layers, d_ff = 64, 4, 2, 256


class BlocTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model)
        )

    def forward(self, x, masque):
        a, _ = self.attn(self.ln1(x), self.ln1(x), self.ln1(x),
                         attn_mask=masque, need_weights=False)
        x = x + a                       # résidu après l'attention
        x = x + self.ffn(self.ln2(x))   # résidu après le FFN
        return x


class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok = nn.Embedding(vocab_size, d_model)
        self.pos = nn.Embedding(block_size, d_model)
        self.blocs = nn.ModuleList([BlocTransformer() for _ in range(n_layers)])
        self.ln_final = nn.LayerNorm(d_model)
        self.tete = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, x):
        B, T = x.shape
        h = self.tok(x) + self.pos(torch.arange(T))                 # (B, T, d_model)
        masque = torch.triu(torch.full((T, T), float("-inf")), diagonal=1)
        for bloc in self.blocs:
            h = bloc(h, masque)                                     # shape préservée
        return self.tete(self.ln_final(h))                         # (B, T, vocab_size)


def fabriquer_batch(gen, taille=16):
    ix = torch.randint(0, len(data) - block_size - 1, (taille,), generator=gen)
    x = torch.stack([data[i : i + block_size] for i in ix])          # (B, T)
    y = torch.stack([data[i + 1 : i + block_size + 1] for i in ix])  # (B, T), décalé de 1
    return x, y


def construire():
    """Fabrique un modèle, un optimizer et un scheduler tout neufs, toujours
    à partir de la même graine : deux appels donnent des poids identiques."""
    torch.manual_seed(42)
    random.seed(42)
    modele = GPT()
    optim = torch.optim.AdamW(modele.parameters(), lr=3e-3)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=60)
    return modele, optim, sched


def entrainer(modele, optim, sched, gen, n_steps, journal):
    for _ in range(n_steps):
        x, y = fabriquer_batch(gen)
        loss = F.cross_entropy(modele(x).view(-1, vocab_size), y.view(-1))
        optim.zero_grad(); loss.backward(); optim.step(); sched.step()
        journal.append(round(loss.item(), 4))
    return journal


# La course de référence de la leçon (60 pas d'un trait). La reprise exacte du
# chapitre lui est identique : elle sert de repère à la validation de l'exercice 2.
modele, optim, sched = construire()
gen = torch.Generator().manual_seed(123)
loss_repris = entrainer(modele, optim, sched, gen, 60, [])
print("mise en place OK | dernière loss de référence :", loss_repris[-1])

### Exercice 1 · Journalisation JSONL et rotation — niveau ●

Écris `journaliser` (une ligne JSON = une mesure, ajoutée en mode `"a"`, *append*)
et complète `rotation` (ne garder que les `garder` checkpoints `etape_*.pt` les plus
récents). Deux trous d'une ligne chacun : `json.dumps` d'un côté, `os.remove` de l'autre.

In [ ]:
def journaliser(chemin, **champs):
    """Ajoute une ligne JSON au fichier de log. Une ligne = une mesure."""
    with open(chemin, "a") as f:
        f.write(json.dumps(champs) + "\n")


def rotation(dossier, garder=3):
    """Ne conserve que les `garder` checkpoints etape_*.pt les plus récents."""
    fichiers = sorted(
        (f for f in os.listdir(dossier) if f.startswith("etape_") and f.endswith(".pt")),
        key=lambda f: int(f[len("etape_"):-len(".pt")]),
    )
    for vieux in fichiers[:-garder]:
        os.remove(os.path.join(dossier, vieux))
    return sorted(
        (f for f in os.listdir(dossier) if f.startswith("etape_")),
        key=lambda f: int(f[len("etape_"):-len(".pt")]),
    )

In [ ]:
# Validation : journalisation et rotation.
os.makedirs("test_log", exist_ok=True)
open("test_log/j.jsonl", "w").close()
journaliser("test_log/j.jsonl", step=0, loss=3.14)
journaliser("test_log/j.jsonl", step=1, loss=2.71)
lignes = [json.loads(l) for l in open("test_log/j.jsonl")]
assert len(lignes) == 2 and lignes[1]["loss"] == 2.71, "le journal JSONL n'a pas 2 lignes valides"

for s in (20, 40, 60, 80):
    open(f"test_log/etape_{s}.pt", "w").close()
restants = rotation("test_log", garder=3)
assert restants == ["etape_40.pt", "etape_60.pt", "etape_80.pt"], f"rotation KO : {restants}"
shutil.rmtree("test_log", ignore_errors=True)
print("Exercice 1 validé : JSONL et rotation OK")

### Exercice 2 · Le checkpoint pauvre, mesurer la dérive — niveau ●

Refais le cas qui échoue de tes mains : sauve **seulement les poids**, reprends avec un
optimizer neuf et une mauvaise graine de RNG, puis mesure la dérive par rapport à la
reprise exacte (`loss_repris`, la liste des 60 loss). Rien ne plante : c'est ça le piège.

In [ ]:
modele, optim, sched = construire()
gen = torch.Generator().manual_seed(123)
loss_casse = entrainer(modele, optim, sched, gen, 30, [])

os.makedirs("checkpoints_demo", exist_ok=True)
torch.save(modele.state_dict(), "checkpoints_demo/poids_seuls.pt")   # les poids, rien d'autre

del modele, optim, sched, gen
modele, optim, sched = construire()          # optimizer NEUF
modele.load_state_dict(torch.load("checkpoints_demo/poids_seuls.pt"))
gen = torch.Generator().manual_seed(999)     # mauvaise graine : autres batchs
loss_casse = entrainer(modele, optim, sched, gen, 30, loss_casse)

In [ ]:
# Validation : la reprise bancale doit s'écarter de la reprise exacte.
derive = round(sum(abs(a - b) for a, b in zip(loss_repris[30:], loss_casse[30:])) / 30, 4)
assert derive > 0.05, (
    f"dérive {derive} trop faible : la reprise bancale devrait s'écarter de la reprise exacte"
)
print(f"Exercice 2 validé : la reprise bancale dérive de {derive} en moyenne "
      "(l'exacte, elle, reste à 0.0)")

### Exercice 3 · Le checkpoint complet et la sauvegarde atomique — niveau ●●

Réécris de mémoire les deux fonctions centrales du chapitre. Les cinq pièces dans le
paquet (poids, optimizer, scheduler, pas, RNG), l'écriture dans un `.tmp` puis le
renommage atomique avec `os.replace`. La validation rejoue le protocole complet
(60 pas d'un trait contre 30 + 30 avec coupure) et exige un écart de 0.0.

In [ ]:
def sauver_checkpoint(chemin, modele, optim, sched, gen, step):
    paquet = {
        "model": modele.state_dict(),
        "optim": optim.state_dict(),
        "sched": sched.state_dict(),
        "step": step,
        "torch_rng": torch.get_rng_state(),   # RNG global de torch
        "gen_rng": gen.get_state(),           # RNG de notre tirage de batchs
        "py_rng": random.getstate(),          # RNG du module random de Python
    }
    tmp = chemin + ".tmp"
    torch.save(paquet, tmp)
    os.replace(tmp, chemin)   # renommage atomique : jamais de fichier à moitié écrit


def charger_checkpoint(chemin, modele, optim, sched, gen):
    paquet = torch.load(chemin)
    modele.load_state_dict(paquet["model"])
    optim.load_state_dict(paquet["optim"])
    sched.load_state_dict(paquet["sched"])
    torch.set_rng_state(paquet["torch_rng"])
    gen.set_state(paquet["gen_rng"])
    random.setstate(paquet["py_rng"])
    return paquet["step"]

In [ ]:
# Validation : le protocole complet, référence contre course coupée/reprise.
os.makedirs("checkpoints_demo", exist_ok=True)

# --- référence : 60 pas d'un trait ---
modele, optim, sched = construire()
gen = torch.Generator().manual_seed(123)
loss_reference = entrainer(modele, optim, sched, gen, 60, [])

# --- même course, coupée en 30 + 30 avec checkpoint ---
modele, optim, sched = construire()
gen = torch.Generator().manual_seed(123)
loss_repris = entrainer(modele, optim, sched, gen, 30, [])
sauver_checkpoint("checkpoints_demo/etape_30.pt", modele, optim, sched, gen, step=30)
del modele, optim, sched, gen

modele, optim, sched = construire()
gen = torch.Generator()
step = charger_checkpoint("checkpoints_demo/etape_30.pt", modele, optim, sched, gen)
loss_repris = entrainer(modele, optim, sched, gen, 30, loss_repris)

ecart_max = max(abs(a - b) for a, b in zip(loss_reference, loss_repris))
assert step == 30, f"le checkpoint doit rendre step=30, pas {step}"
assert ecart_max == 0.0, (
    f"écart max {ecart_max} != 0 : ton checkpoint est incomplet "
    "(optimizer, scheduler ou RNG manquant)"
)
print("Exercice 3 validé : reprise EXACTE, écart maximum =", ecart_max)

### Exercice 4 · La boucle qui survit — niveau ●●●

Assemble tout : la boucle **reprend** depuis le dernier checkpoint s'il existe,
**journalise** chaque mesure, **sauve** un checkpoint complet tous les `tous_les` pas
et fait la **rotation** (`garder=3`). Les deux trous sont au cœur de la boucle :
la sauvegarde et la rotation, au bon moment et avec le bon numéro de pas.

In [ ]:
def boucle_qui_survit(dossier, total_steps=60, tous_les=20):
    os.makedirs(dossier, exist_ok=True)
    modele, optim, sched = construire()
    gen = torch.Generator().manual_seed(123)
    log = os.path.join(dossier, "journal.jsonl")

    derniers = sorted(
        (f for f in os.listdir(dossier) if f.startswith("etape_") and f.endswith(".pt")),
        key=lambda f: int(f[len("etape_"):-len(".pt")]),
    )
    depart = 0
    if derniers:
        depart = charger_checkpoint(os.path.join(dossier, derniers[-1]),
                                    modele, optim, sched, gen)
    else:
        open(log, "w").close()

    derniere_loss = None
    for step in range(depart, total_steps):
        x, y = fabriquer_batch(gen)
        loss = F.cross_entropy(modele(x).view(-1, vocab_size), y.view(-1))
        optim.zero_grad(); loss.backward(); optim.step(); sched.step()
        derniere_loss = loss.item()
        journaliser(log, step=step, loss=round(derniere_loss, 4))
        if (step + 1) % tous_les == 0:
            sauver_checkpoint(os.path.join(dossier, f"etape_{step + 1}.pt"),
                              modele, optim, sched, gen, step + 1)
            rotation(dossier, garder=3)
    return depart, derniere_loss

In [ ]:
# Validation : la boucle doit reprendre toute seule au bon pas.
shutil.rmtree("run_survie", ignore_errors=True)
depart1, _ = boucle_qui_survit("run_survie", total_steps=40)   # part de 0
depart2, _ = boucle_qui_survit("run_survie", total_steps=60)   # doit reprendre à 40
assert depart1 == 0, f"premier appel : départ {depart1} != 0"
assert depart2 == 40, f"second appel : la reprise devrait partir de 40, pas {depart2}"
print("Exercice 4 validé : la boucle reprend toute seule au bon pas")

# Nettoyage final.
for d in ("checkpoints_demo", "run_survie"):
    shutil.rmtree(d, ignore_errors=True)